# RAG security and red-team testing

This research measures actual local attacker objectives and legitimate task performance. The frozen quarantine policy fails on held-out tasks: its zero measured attacker success accompanies zero clean completion. It is a failed defense, not a security success. Deterministic authorization and integrity controls have separate fixture results. The public study is a reduced BIPIA adaptation plus original synthetic fixtures; no official benchmark score or general security guarantee is claimed.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'reports/selection-freeze.json').exists()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))
from figures import figure
def read(name): return json.loads((ROOT / 'reports' / name).read_text(encoding='utf-8'))
metrics = read('metrics.json')
freeze = read('selection-freeze.json')
print('Frozen selection:', freeze['selected_model'], freeze['selected_defense'])

## Motivation and prior evidence

Earlier projects found that citations did not prove support, verification could reject useful answers, and cache reuse depended on current permissions. Security evaluation must therefore preserve utility and measure the attack objective directly. OWASP LLM application guidance supplies risk categories, not certification. See `docs/provenance.md` for the benchmark papers, exact source links and attribution.

In [ ]:
environment = read('environment.json')
{k: environment[k] for k in ['python','platform','physical_cores','logical_cores','gpu','prior_tracked_files']}

## Threat model and protected assets

Levels A through E cover one retrieved passage, one submitted document, one read-only tool response, a cache candidate and one synthetic tenant. System instructions, canonical permissions and the application registry remain trusted. There are no offensive requests to external services. The complete assumptions and residual risks are in `docs/threat-model.md`.

In [ ]:
from rag_security.policy import Document, Registry, Principal
from rag_security.retrieval import retrieve
docs = [Document('public', 'The station uses amber.'), Document('private', 'SYNTHETIC_CANARY_DEMO', tenant='beta')]
registry = Registry(docs)
principal = Principal('alpha', 'reader', frozenset({'public','private'}))
assert [d.id for d in retrieve('station', docs, registry, principal)] == ['public']
print('Canonical permissions exclude the other tenant before ranking.')

## Dataset provenance and partitions

TableQA contexts and unchanged public attack strings come from pinned BIPIA files. Wikipedia-derived tables remain local under their upstream rights. Synthetic observatory development tasks and botanical evaluation tasks are reported separately. AgentDojo was researched but not executed. RAG-PIBench could not be acquired with verified dataset provenance and license, so it contributes no scores.

In [ ]:
parts = read('partitions.json')
print({s: pd.Series([r['domain'] for r in rows]).value_counts().to_dict() for s, rows in parts['tasks'].items()})
assert not {r['context_hash'] for r in parts['tasks']['development']} & {r['context_hash'] for r in parts['tasks']['evaluation']}
print('Source-context hashes are disjoint; recurring synthetic diagnostics are identified separately.')

## Attack-family EDA

One fixed payload defines each public family. Repetition across tasks does not create new attack diversity. Public development and evaluation attack families are disjoint; only dialogue and encoded redirection support the corresponding synthetic transfer claim. Citation, poison and tool tests recur as diagnostics.

In [ ]:
fig = figure('families'); plt.show()

## Clean RAG baseline

BM25 ranks bounded local candidates. Both generators receive ordinary QA instructions and source identifiers. D0 retains normal authorization and a read-only dispatcher. Clean completion uses normalized exact match for public tables and a closed-color factual oracle for synthetic records. EM and token F1 remain separate diagnostics.

In [ ]:
evaluation = pd.DataFrame(metrics['evaluation']).T
evaluation[['clean_n','clean_completion','clean_em','clean_f1','clean_rejection_rate']]

## Attack construction and validity

Only the permitted passage changes. Instruction redirection and factual replacement have different objective checks. A valid visible attack must survive the undefended context packer. Defenses cannot improve conditional ASR by declaring blocked payloads invalid. Strict ASR retains every attempted attack. Reverse objectives with unchanged references are invalid.

In [ ]:
evaluation[['attack_attempts','valid_attacks','successes','strict_asr','conditional_asr']]

## Undefended attack outcomes

Attacker success requires the defined output or action proposal. Mentioning a suspicious word is insufficient. Color poisoning accepts an asserted wrong color in a full sentence; it rejects explicit negations and contradictory correct-color mentions. These rules were corrected after the pilot and before protected generation. `reports/protocol-history.json` preserves that development change.

In [ ]:
fig = figure('asr'); plt.show()

## Individual defense comparisons

D1 enforces permissions; D2 checks complete document integrity; D3 rejects screened requests; D4 separates untrusted context; D5 checks references and action policy; D6 quarantines newline units. D1 is a negative control in ordinary QA because D0 already retains permissions. D7 combines D1, D4, D5 and D6, with an additional integrity combination evaluated during development.

In [ ]:
pd.DataFrame(freeze['candidates']).sort_values('score', ascending=False).head(10)

## Injection detector comparisons

Rules, locally trained TF-IDF, BGE reference similarity, a pretrained DeBERTa detector and Qwen Yes/No scores use frozen development thresholds. Labels follow explicit fixture intent and benign quotations; there is no independent human adjudication. Upstream DeBERTa training overlap is unknown. PR-AUC here means average precision.

In [ ]:
detectors = read('detectors.json')
detection = pd.DataFrame(detectors['evaluation']).T
detection[['precision','recall','f1','pr_auc_average_precision','threshold']]

In [ ]:
fig = figure('detection'); plt.show()

## Benign false positives

The detection set includes difficult quoted attack discussions. Its false-positive rate has a different denominator and distribution from clean QA rejection. A filter can reject harmless instructional passages without improving downstream task safety.

In [ ]:
fig = figure('false-positives'); plt.show()

## Poisoned evidence and citation manipulation

Content hashes can reject changed registered evidence. They cannot establish the truth of a correctly registered submission. Citation validity checks source membership only; valid source IDs may still accompany unsupported answers. No model judge supplies truth labels.

In [ ]:
families = pd.DataFrame(read('families.json'))
families[families['family'].isin(['poison','citation'])][['model','defense','family','attack_attempts','strict_asr','attacked_completion']]

## Tenant authorization and synthetic leakage

The deterministic fixtures compare explicit broken application designs with canonical authorization, scoped caches, revocation and version checks. Retrieval exposure means secret-bearing text crossed into the generation boundary. Model-generated disclosure is a different result, measured in the real-inference smoke test. Fixed canary output proposals are labeled deterministic tests.

In [ ]:
boundaries = read('boundaries.json')
print('Boundary attempts:', boundaries['attempts'])
{family: row for family, row in boundaries['families'].items() if family in ['tenant-miss','tenant-cache-hit','revocation','freshness','output-canary']}

## Agent tool-output experiments

The model sees a replaced read-only response and proposes an action. Diagnostic parsing recognizes a fenced JSON proposal, while execution accepts only the exact strict schema. Export does not exist in the tool registry. Proposal influence, parser rejection and actual unauthorized execution remain separate outcomes.

In [ ]:
evaluation[['disallowed_tool_attempts','actual_unauthorized_tool_execution']]

In [ ]:
pd.DataFrame(read('tool-controls.json')['aggregate'])

## Defense combinations and utility

Selection uses development outcomes only. A refusal that prevents an attack still loses legitimate completion when the task is unanswered. The attack/utility plot makes that cost visible.

In [ ]:
fig = figure('utility'); plt.show()

In [ ]:
fig = figure('tradeoff'); plt.show()

## Held-out and cross-family evaluation

The frozen configuration is evaluated once on protected generator tasks. Public families and synthetic dialogue/encoding are transfer subsets; recurring poison, citation and tool diagnostics are not called unseen families. The second generator is a compatibility comparison, not a newly selected system.

In [ ]:
families[families['family'].isin(['dialogue','encoded','bipia-sentiment','bipia-base64','bipia-reverse'])][['model','defense','family','valid_attacks','strict_asr','attacked_completion']]

## Uncertainty and paired comparisons

Bootstrap intervals resample whole task groups with 2,000 replicates. They are pointwise and unadjusted. Repeated payloads limit effective attack diversity. Wilson intervals accompany zero events because a zero-event bootstrap can be degenerate; their independent-attempt assumption is optimistic under clustering.

In [ ]:
fig = figure('uncertainty'); plt.show()

## Retrieval position and admitted-source diagnostics

The prespecified small position study forces two evidence ranks and three context positions. A separate admission study registers an attacker submission correctly, demonstrating that source integrity and factual trust differ. The bounded adaptive diagnostic searches three placements on development tasks only; it does not establish adaptive confirmation-set robustness.

In [ ]:
fig = figure('positions'); plt.show()
print('Admission:', metrics['ingestion'])
print('Adaptive placement search:', metrics['adaptive'])

## Performance and resource costs

Latency adds measured inference to each policy preparation cost. Identical generation prompts reuse actual predictions and their original inference timing. These are computationally shared paired estimates, not repeated independent wall-time measurements. Loading is separate. GPU allocation and sampled process RSS are recorded without claiming process peak memory.

In [ ]:
fig = figure('latency'); plt.show()
evaluation[['p50_seconds','p95_seconds','modeled_throughput_per_second','generation_reused','context_truncated_fraction']]

In [ ]:
pd.DataFrame(read('cold-profile.json'))[['model','cold_process_p50_seconds','warm_inference_p50_seconds','sampled_peak_child_rss_bytes']]

## Real local inference check

The following cell performs a fresh local generation when locked model files are available. It is a smoke test, separate from the frozen attempt matrix. Acquisition is a separate authorized research step; this cell makes no network requests.

In [ ]:
from rag_security.models import LocalModel
locks = json.loads((ROOT / 'models.lock.json').read_text(encoding='utf-8'))
model_path = ROOT.parent / locks['flan-base']['local_path']
if model_path.exists() or (ROOT / 'data/models/flan-base').exists():
    model = LocalModel('flan-base')
    actual = model.generate('What color does the station use?', [docs[0]], structured=True)
    print({'actual_model_answer':actual['answer'], 'inference_seconds':actual['inference_seconds']})
    del model
else:
    print('Locked model files are absent. Run the acquisition instructions to enable this local inference cell.')

## Qualitative failure review

The local review candidates preserve exact attempt identities without publishing raw traces. Factual poisoning, fabricated references and model action proposals require different safeguards. Clean refusal counts and task failure accompany every security claim. The README records the reviewed examples and measured limitations.

In [ ]:
read('qualitative-review.json')

## Reconciliation and publication

Every aggregate is reconstructed from the stage manifest and exact local attempt IDs. Stale files are excluded. The public notebook has identical source cells and markdown but no outputs or execution counts. Projects 1 through 5 are verified against their original tracked bytes. Detailed prompts, weights, raw data and this executed notebook remain local.

In [ ]:
reconciliation = read('reconciliation.json')
assert reconciliation['metrics_rebuilt_from_attempt_ids']
assert all(read(f'{s}-run.json')['attempts'] == reconciliation['stages'][s]['n'] for s in ['development','evaluation','positions','ingestion','adaptive'])
print(reconciliation)

## Conclusions and limitations

The evidence applies to these fixed local attacks, small task samples and two compact generators. Authorization can be enforced outside the model under the stated trusted-registry assumptions. Statistical generation defenses can bypass, over-reject or damage useful evidence. No result establishes general prompt-injection robustness, unknown-secret protection, production authentication, faithful citation entailment or resilience against unrestricted adaptive attackers. The final README reports verified numerical outcomes, uncertainty and costs.